# Notebook 04 - Orquestador Con Routing

Este notebook solo crea el orquestador. Los agentes especializados ya vienen del notebook 03.

### Descripcion de la celda 1Carga configuracion, valida variables y prepara el contexto.

In [63]:
# Celda 2: ejecuta este paso del notebook y prepara resultados para el siguiente.import json
import os
from pathlib import Path
import requests
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

def load_env_file(path='config.env'):
    env_map = {}
    p = Path(path)
    if not p.exists():
        return env_map
    for raw in p.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        env_map[k.strip()] = v.strip().strip("\"").strip("'")
    return env_map

env_file = load_env_file('config.env')
cfg_path = Path('workshop_config.json')
cfg = {}
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text(encoding='utf-8'))

def env(name, default=None, required=False):
    v = os.getenv(name, env_file.get(name, default))
    if required and (v is None or str(v).strip() == ''):
        raise RuntimeError(f'Falta variable requerida: {name}')
    return v

RESPONSES_API_VERSION = '2025-03-01-preview'
FOUNDRY_AGENTS_API_VERSION = env('FOUNDRY_AGENTS_API_VERSION', cfg.get('foundry_agents_api_version', '2025-05-15-preview'))

AZURE_OPENAI_ENDPOINT = env('AZURE_OPENAI_ENDPOINT', cfg.get('azure_openai_endpoint', ''), required=True)
MODEL = env('MODEL_DEPLOYMENT_NAME', cfg.get('model_deployment_name', 'gpt-4o'))
API_VERSION = env('AZURE_OPENAI_API_VERSION', env('API_VERSION', cfg.get('api_version', '2025-01-01-preview')))
USER_ALIAS = env('USER_ALIAS', cfg.get('user_alias', 'user01')).strip()
PREFIX = env('RESOURCE_PREFIX', cfg.get('resource_prefix', f'contoso-{USER_ALIAS}'))

assert cfg.get('especialistas_ready'), 'Ejecuta primero notebook 03'

SEARCH_SERVICE_NAME = cfg.get('ai_search', {}).get('service_name', f"{USER_ALIAS}srch".replace('-', '')[:48])
SEARCH_INDEX = cfg.get('ai_search', {}).get('index_name', f"{PREFIX}-kb".lower()[:128])
SEARCH_ENDPOINT = cfg.get('ai_search', {}).get('endpoint', f"https://{SEARCH_SERVICE_NAME}.search.windows.net")
SEARCH_API_VERSION = cfg.get('ai_search', {}).get('api_version', '2024-07-01')
SEARCH_ADMIN_KEY = cfg.get('ai_search', {}).get('admin_key', env('AZURE_SEARCH_ADMIN_KEY', ''))
USE_VECTOR_SEARCH = bool(cfg.get('ai_search', {}).get('use_vector_search', False))
EMBEDDING_DEPLOYMENT = env('EMBEDDING_MODEL_DEPLOYMENT', cfg.get('ai_search', {}).get('embedding_model_deployment', cfg.get('embedding_model_deployment', '')))

FUNCTION_BASE_URL = env('FUNCTION_BASE_URL', cfg.get('function_app', {}).get('base_url', f'https://{USER_ALIAS}contosofunc.azurewebsites.net/api'))
FOUNDRY_PROJECT_ENDPOINT = env('FOUNDRY_PROJECT_ENDPOINT', cfg.get('foundry_project_endpoint', ''), required=True).rstrip('/')

ESP = cfg.get('agentes_especializados', {
    'conocimiento': {'name': f'{PREFIX}-esp-conocimiento'},
    'cliente': {'name': f'{PREFIX}-esp-cliente'},
    'soporte': {'name': f'{PREFIX}-esp-soporte'}
})
FOUNDRY_AI_SEARCH_CONNECTION = cfg.get('foundry_connections', {}).get('ai_search', f'{PREFIX}-ai-search')
FOUNDRY_AI_SEARCH_CONNECTION_ID = cfg.get('foundry_connections', {}).get('ai_search_id', env('FOUNDRY_AI_SEARCH_CONNECTION_ID', ''))
FOUNDRY_FUNCTION_CONNECTION = cfg.get('foundry_connections', {}).get('function_api', f'{PREFIX}-function-api')
FOUNDRY_FUNCTION_CONNECTION_ID = cfg.get('foundry_connections', {}).get('function_api_id', env('FOUNDRY_FUNCTION_CONNECTION_ID', FOUNDRY_FUNCTION_CONNECTION))

assert FOUNDRY_AI_SEARCH_CONNECTION_ID, 'Falta foundry_connections.ai_search_id. Ejecuta Notebook 03 para regenerar conexiones.'

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, 'https://cognitiveservices.azure.com/.default')
client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version=RESPONSES_API_VERSION
)

print('OK contexto orquestador (workshop_config + env)')
print('Foundry endpoint:', FOUNDRY_PROJECT_ENDPOINT)
print('Vector search habilitado:', USE_VECTOR_SEARCH)

OK contexto orquestador (Responses API)
Responses API version: 2025-03-01-preview
Vector search habilitado: True
Foundry endpoint: https://ai-williamta-9386.services.ai.azure.com/api/projects/ai-williamta-9386-project


### Descripcion de la celda 2Crea o valida Azure AI Search y deja el indice listo.

In [64]:
# Celda 3: ejecuta este paso del notebook y prepara resultados para el siguiente.def _foundry_headers() -> dict:
    token = credential.get_token('https://ai.azure.com/.default').token
    return {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

def _foundry_list_agents() -> dict:
    r = requests.get(
        f"{FOUNDRY_PROJECT_ENDPOINT}/agents?api-version={FOUNDRY_AGENTS_API_VERSION}",
        headers=_foundry_headers(),
        timeout=30
    )
    if r.status_code != 200:
        raise RuntimeError(f'No se pudo listar agentes Foundry: {r.status_code} {r.text}')
    return {a['id']: a for a in r.json().get('data', [])}

def _delete_foundry_agent(agent_name: str) -> None:
    requests.delete(
        f"{FOUNDRY_PROJECT_ENDPOINT}/agents/{agent_name}?api-version={FOUNDRY_AGENTS_API_VERSION}",
        headers=_foundry_headers(),
        timeout=30
    )

def ensure_foundry_prompt_agent(agent_name: str, instructions: str, tools: list | None = None, metadata: dict | None = None) -> dict:
    tools = tools or []
    metadata = metadata or {}
    existing = _foundry_list_agents()
    if agent_name in existing:
        latest = existing[agent_name].get('versions', {}).get('latest', {})
        existing_instructions = latest.get('definition', {}).get('instructions', '')
        existing_tools = latest.get('definition', {}).get('tools', [])
        existing_meta = latest.get('metadata', {})
        tools_equal = json.dumps(existing_tools, sort_keys=True) == json.dumps(tools, sort_keys=True)
        instructions_equal = existing_instructions == instructions
        metadata_equal = json.dumps(existing_meta, sort_keys=True) == json.dumps(metadata, sort_keys=True)
        if tools_equal and instructions_equal and metadata_equal:
            return {'id': agent_name, 'name': existing[agent_name].get('name', agent_name), 'created': False}
        _delete_foundry_agent(agent_name)

    payload = {
        'name': agent_name,
        'metadata': metadata,
        'definition': {
            'kind': 'prompt',
            'model': MODEL,
            'instructions': instructions,
            'tools': tools
        }
    }
    r = requests.post(
        f"{FOUNDRY_PROJECT_ENDPOINT}/agents?api-version={FOUNDRY_AGENTS_API_VERSION}",
        headers=_foundry_headers(),
        json=payload,
        timeout=30
    )
    if r.status_code not in (200, 201):
        raise RuntimeError(f'No se pudo crear agente Foundry {agent_name}: {r.status_code} {r.text}')
    data = r.json()
    return {'id': data['id'], 'name': data.get('name', data['id']), 'created': True}

def search_knowledge(query: str) -> str:
    url = f"{SEARCH_ENDPOINT}/indexes/{SEARCH_INDEX}/docs/search?api-version={SEARCH_API_VERSION}"
    body = {'top': 3, 'select': 'source,content'}
    if USE_VECTOR_SEARCH and EMBEDDING_DEPLOYMENT:
        qv = client.embeddings.create(model=EMBEDDING_DEPLOYMENT, input=query).data[0].embedding
        body['vectorQueries'] = [{
            'kind': 'vector',
            'vector': qv,
            'fields': 'contentVector',
            'k': 3
        }]
    else:
        body['search'] = query

    r = requests.post(url, headers={'Content-Type': 'application/json', 'api-key': SEARCH_ADMIN_KEY}, json=body, timeout=30)
    if r.status_code != 200:
        return json.dumps({'error': r.text})
    return json.dumps(r.json().get('value', []), ensure_ascii=False)

def call_business_function(function_name: str, payload: dict) -> str:
    r = requests.post(f"{FUNCTION_BASE_URL}/{function_name}", json=payload, timeout=30)
    return r.text

ESPECIALISTAS = {
    'conocimiento': {
        'name': ESP['conocimiento']['name'],
        'instructions': 'Especialista de conocimiento. Debes llamar tool_search_knowledge en cada consulta y responder solo con datos devueltos por esa tool. Si el resultado incluye horarios o canales, listalos explicitamente.',
        'tools': [
            {
                'type': 'function',
                'name': 'tool_search_knowledge',
                'description': 'Busca conocimiento en AI Search',
                'parameters': {
                    'type': 'object',
                    'properties': {'query': {'type': 'string'}},
                    'required': ['query']
                }
            }
        ]
    },
    'cliente': {
        'name': ESP['cliente']['name'],
        'instructions': 'Especialista de cliente. Debes usar solo tool_buscar_cliente o tool_listar_productos y responder con la data obtenida, sin inventar.',
        'tools': [
            {
                'type': 'function',
                'name': 'tool_buscar_cliente',
                'description': 'Busca cliente por cedula',
                'parameters': {
                    'type': 'object',
                    'properties': {'cedula': {'type': 'string'}},
                    'required': ['cedula']
                }
            },
            {
                'type': 'function',
                'name': 'tool_listar_productos',
                'description': 'Lista productos disponibles',
                'parameters': {
                    'type': 'object',
                    'properties': {},
                    'required': []
                }
            }
        ]
    },
    'soporte': {
        'name': ESP['soporte']['name'],
        'instructions': 'Especialista de soporte. Debes usar tool_consultar_tickets y responder con el resultado exacto.',
        'tools': [
            {
                'type': 'function',
                'name': 'tool_consultar_tickets',
                'description': 'Consulta tickets por cliente_id',
                'parameters': {
                    'type': 'object',
                    'properties': {'cliente_id': {'type': 'string'}},
                    'required': ['cliente_id']
                }
            }
        ]
    }
}

def _tool_dispatch(fn: str, args: dict) -> str:
    if fn == 'tool_search_knowledge':
        return search_knowledge(args['query'])
    if fn == 'tool_buscar_cliente':
        return call_business_function('buscar_cliente', {'cedula': args['cedula']})
    if fn == 'tool_listar_productos':
        return call_business_function('listar_productos_disponibles', {})
    if fn == 'tool_consultar_tickets':
        return call_business_function('consultar_tickets_cliente', {'cliente_id': args['cliente_id']})
    return json.dumps({'error': f'tool no permitida: {fn}'})

def _run_response_agent(instructions: str, tools: list, prompt: str) -> str:
    response = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=prompt,
        tools=tools,
        tool_choice='required'
    )

    while True:
        function_calls = [o for o in response.output if getattr(o, 'type', '') == 'function_call']
        if not function_calls:
            return response.output_text or ''

        tool_outputs = []
        for call in function_calls:
            args = json.loads(call.arguments or '{}')
            output = _tool_dispatch(call.name, args)
            tool_outputs.append({
                'type': 'function_call_output',
                'call_id': call.call_id,
                'output': output
            })

        response = client.responses.create(
            model=MODEL,
            previous_response_id=response.id,
            input=tool_outputs
        )

def _run_especialista(tipo: str, consulta: str) -> str:
    esp = ESPECIALISTAS[tipo]
    return _run_response_agent(esp['instructions'], esp['tools'], consulta)

def route_conocimiento(consulta: str) -> str:
    return _run_especialista('conocimiento', consulta)

def route_cliente(consulta: str) -> str:
    return _run_especialista('cliente', consulta)

def route_soporte(consulta: str) -> str:
    return _run_especialista('soporte', consulta)

### Descripcion de la celda 3Crea o valida Azure AI Search y deja el indice listo.

In [67]:
# Celda 4: ejecuta este paso del notebook y prepara resultados para el siguiente.orq_tools = [
  {
    'type': 'function',
    'name': 'route_conocimiento',
    'description': 'Usa esta ruta para horarios, políticas, FAQ, canales y conocimiento general',
    'parameters': {
      'type': 'object',
      'properties': {'consulta': {'type': 'string'}},
      'required': ['consulta']
    }
  },
  {
    'type': 'function',
    'name': 'route_cliente',
    'description': 'Usa esta ruta para datos de cliente por cedula y catálogo de productos',
    'parameters': {
      'type': 'object',
      'properties': {'consulta': {'type': 'string'}},
      'required': ['consulta']
    }
  },
  {
    'type': 'function',
    'name': 'route_soporte',
    'description': 'Usa esta ruta para tickets y casos de soporte',
    'parameters': {
      'type': 'object',
      'properties': {'consulta': {'type': 'string'}},
      'required': ['consulta']
    }
  }
]

ORQUESTADOR_INSTRUCTIONS = (
    'Eres el orquestador. Siempre debes elegir exactamente una ruta y delegar. '
    'Reglas de routing: '
    '1) horarios, FAQ, politicas, canales, informacion general -> route_conocimiento; '
    '2) cedula, cliente, productos, credito -> route_cliente; '
    '3) ticket, soporte, caso -> route_soporte. '
    'Tu respuesta final debe estar basada en lo que devolvio el especialista, sin inventar informacion.'
)
ORQUESTADOR_NAME = f"{PREFIX}-orquestador"
assert FOUNDRY_AI_SEARCH_CONNECTION_ID, 'Falta foundry_connections.ai_search_id. Ejecuta Notebook 03 para regenerar conexiones.'

orq_foundry_tools = [
  {
    'type': 'azure_ai_search',
    'name': 'ai_search_kb',
    'description': 'Consulta el indice de conocimiento para FAQ, horarios y politicas.',
    'azure_ai_search': {
      'indexes': [
        {
          'project_connection_id': FOUNDRY_AI_SEARCH_CONNECTION_ID,
          'index_name': SEARCH_INDEX,
          'query_type': 'simple',
          'top_k': 5,
          'endpoint': SEARCH_ENDPOINT,
          'authentication': {'type': 'api_key', 'key': SEARCH_ADMIN_KEY}
        }
      ],
      'indexConfiguration': {
        'projectConnectionId': FOUNDRY_AI_SEARCH_CONNECTION_ID,
        'indexName': SEARCH_INDEX
      }
    }
  },
  {
    'type': 'openapi',
    'name': 'business_api_cliente',
    'description': 'API de cliente para buscar por cedula y listar productos disponibles.',
    'openapi': {
      'name': 'business_api_cliente',
      'spec': {
        'openapi': '3.0.1',
        'info': {'title': 'Contoso Cliente API', 'version': '1.0.0'},
        'servers': [{'url': FUNCTION_BASE_URL}],
        'paths': {
          '/buscar_cliente': {
            'post': {
              'operationId': 'buscar_cliente',
              'description': 'Busca cliente por cedula',
              'requestBody': {
                'required': True,
                'content': {
                  'application/json': {
                    'schema': {
                      'type': 'object',
                      'properties': {'cedula': {'type': 'string'}},
                      'required': ['cedula']
                    }
                  }
                }
              },
              'responses': {'200': {'description': 'OK'}}
            }
          },
          '/listar_productos_disponibles': {
            'post': {
              'operationId': 'listar_productos_disponibles',
              'description': 'Lista productos disponibles',
              'responses': {'200': {'description': 'OK'}}
            }
          }
        }
      },
      'auth': {'type': 'connection', 'connection_id': FOUNDRY_FUNCTION_CONNECTION_ID}
    }
  },
  {
    'type': 'openapi',
    'name': 'business_api_soporte',
    'description': 'API de soporte para consultar tickets de un cliente.',
    'openapi': {
      'name': 'business_api_soporte',
      'spec': {
        'openapi': '3.0.1',
        'info': {'title': 'Contoso Soporte API', 'version': '1.0.0'},
        'servers': [{'url': FUNCTION_BASE_URL}],
        'paths': {
          '/consultar_tickets_cliente': {
            'post': {
              'operationId': 'consultar_tickets_cliente',
              'description': 'Consulta tickets por cliente_id',
              'requestBody': {
                'required': True,
                'content': {
                  'application/json': {
                    'schema': {
                      'type': 'object',
                      'properties': {'cliente_id': {'type': 'string'}},
                      'required': ['cliente_id']
                    }
                  }
                }
              },
              'responses': {'200': {'description': 'OK'}}
            }
          }
        }
      },
      'auth': {'type': 'connection', 'connection_id': FOUNDRY_FUNCTION_CONNECTION_ID}
    }
  }
]

foundry_orq_name = cfg.get('foundry_agents', {}).get('orquestador', {}).get('name', f"{PREFIX}-orquestador-foundry")
orq_foundry = ensure_foundry_prompt_agent(
    foundry_orq_name,
    ORQUESTADOR_INSTRUCTIONS,
    tools=orq_foundry_tools,
    metadata={
        'tool_mode': 'native',
        'routing_mode': 'single_agent_multi_tools',
        'ai_search_connection': FOUNDRY_AI_SEARCH_CONNECTION,
        'function_connection': FOUNDRY_FUNCTION_CONNECTION
    }
)
estado = 'creado/actualizado' if orq_foundry['created'] else 'ya existia'
print('Orquestador preparado:', ORQUESTADOR_NAME)
print(f"Foundry orquestador: {orq_foundry['name']} ({estado})")

Orquestador preparado: contoso-user01-orquestador
Foundry orquestador: contoso-user01-orquestador-foundry (creado/actualizado)


### Descripcion de la celda 4Ejecuta el flujo de orquestacion y enrutamiento.

In [50]:
# Celda 5: ejecuta este paso del notebook y prepara resultados para el siguiente.def conversar_orquestador(mensaje: str) -> str:
    response = client.responses.create(
        model=MODEL,
        instructions=ORQUESTADOR_INSTRUCTIONS,
        input=mensaje,
        tools=orq_tools,
        tool_choice='required'
    )

    while True:
        function_calls = [o for o in response.output if getattr(o, 'type', '') == 'function_call']
        if not function_calls:
            return response.output_text or ''

        outs = []
        for call in function_calls:
            args = json.loads(call.arguments or '{}')
            if call.name == 'route_conocimiento':
                output = route_conocimiento(**args)
            elif call.name == 'route_cliente':
                output = route_cliente(**args)
            elif call.name == 'route_soporte':
                output = route_soporte(**args)
            else:
                output = 'Ruta no soportada'
            outs.append({
                'type': 'function_call_output',
                'call_id': call.call_id,
                'output': output
            })

        response = client.responses.create(
            model=MODEL,
            previous_response_id=response.id,
            input=outs
        )

print('Motor del orquestador listo (Responses API)')

Motor del orquestador listo (Responses API)


### Descripcion de la celda 5Ejecuta el flujo de orquestacion y enrutamiento.

In [51]:
# Celda 6: ejecuta este paso del notebook y prepara resultados para el siguiente.tests = [
    'Hola, cuales son los horarios de atencion?',
    'Consulta el cliente con cedula 1234567890 y resume su estado.',
    'Muestrame los tickets del cliente CLI-1002' 
]

for i, t in enumerate(tests, start=1):
    print(f'\n=== Prueba {i} ===')
    print('Pregunta:', t)
    print('Respuesta:', conversar_orquestador(t))


=== Prueba 1 ===
Pregunta: Hola, cuales son los horarios de atencion?
Respuesta: Los horarios de atención varían según el servicio:

- **Sucursales físicas**: Lunes a viernes de 8:00 AM a 4:00 PM y sábados de 9:00 AM a 12:00 PM.
- **Línea telefónica**: Disponible de lunes a sábado de 7:00 AM a 8:00 PM.
- **Canales digitales** (como chat o asistente virtual): Disponibles las 24 horas, todos los días.

Si necesitas información sobre un lugar o servicio en particular, indícame más detalles para ayudarte mejor. 😊

=== Prueba 2 ===
Pregunta: Consulta el cliente con cedula 1234567890 y resume su estado.
Respuesta: La cliente con cédula **1234567890**, María García López, tiene un estado positivo en general:

- Cumple al día con sus productos financieros, tanto un **crédito personal** con saldo pendiente de $8,500,000 COP como una **tarjeta de crédito** con cupo disponible de $6,800,000 COP.
- Buen score crediticio de **750** y estabilidad laboral con **48 meses de antigüedad**.
- No tiene s

### Descripcion de la celda 6Actualiza y guarda el estado para los siguientes notebooks.

In [66]:
# Celda 7: ejecuta este paso del notebook y prepara resultados para el siguiente.state = {}
config_path = Path('workshop_config.json')
if config_path.exists():
    state = json.loads(config_path.read_text(encoding='utf-8'))

state.update({
    'azure_openai_endpoint': AZURE_OPENAI_ENDPOINT,
    'api_version': API_VERSION,
    'model_deployment_name': MODEL,
    'resource_prefix': PREFIX,
    'foundry_project_endpoint': FOUNDRY_PROJECT_ENDPOINT,
    'orquestador': {'name': ORQUESTADOR_NAME},
    'foundry_agents': {
        **state.get('foundry_agents', {}),
        'orquestador': {'id': orq_foundry['id'], 'name': orq_foundry['name']}
    },
    'orquestador_runtime': 'responses_api',
    'orquestador_ready': True
})
config_path.write_text(json.dumps(state, indent=2), encoding='utf-8')
print('OK orquestador guardado en workshop_config.json')

OK orquestador guardado


### Descripcion de la celda 7Ejecuta el flujo de orquestacion y enrutamiento.

In [53]:
# Validacion rapida de routing (salida resumida)
cases = [
    ('conocimiento', 'Horarios de atencion y canales digitales'),
    ('cliente', 'Consulta el cliente con cedula 1234567890'),
    ('soporte', 'Muestrame los tickets del cliente CLI-1002')
]

for label, prompt in cases:
    ans = conversar_orquestador(prompt)
    resumen = ans.splitlines()[0] if ans else '(sin respuesta)'
    print(f'{label}: {resumen}')

conocimiento: Aquí tienes información general sobre **horarios de atención** y **canales digitales** para servicios al cliente:
cliente: La información consultada sobre el cliente con cédula **1234567890** incluye datos generales, productos activos, y el estado actual de sus cuentas. Si necesitas realizar alguna operación, como una actualización de datos, consulta específica, o análisis financiero, ¡hazme saber!
soporte: Aquí está la información del ticket del cliente CLI-1002. ¿Te gustaría actualizar el estado del ticket, asignarlo a otro agente o consultar algún otro detalle?
